In [ ]:
!pip install gpxpy
import pandas as pd
import gpxpy
import re
from datetime import timezone, timedelta, datetime
import requests

accel_data  = "https://raw.githubusercontent.com/scramer27/road_surface_classification/main/acceleration_2026-04-19_19-10-51.csv"
label_data = "https://raw.githubusercontent.com/scramer27/road_surface_classification/main/labels_2026-04-19_18-47-48.csv"
response = requests.get("https://raw.githubusercontent.com/scramer27/road_surface_classification/main/19-Apr-2026-1910.gpx")

accel  = pd.read_csv(accel_data)
labels = pd.read_csv(label_data)
gpx = gpxpy.parse(response.text)
time_zone_diff = -4
acceleration_file = "acceleration_2026-04-19_19-10-51.csv"



In [ ]:
# get accel start time from filename
match = re.search(r'(\d{4}-\d{2}-\d{2})_(\d{2}-\d{2}-\d{2})', acceleration_file)
date_str, time_str = match.group(1), match.group(2).replace('-', ':')
accel_end = datetime.strptime(f"{date_str} {time_str}", "%Y-%m-%d %H:%M:%S")
accel_end = accel_end.replace(tzinfo=timezone.utc) - timedelta(hours=time_zone_diff)


In [ ]:
accel_start = accel_end - timedelta(seconds=float(accel['time'].max()))
print(f"accel start: {accel_start}  end: {accel_end}  rows: {len(accel)}")


accel start: 2026-04-19 22:49:08.361000+00:00  end: 2026-04-19 23:10:51+00:00  rows: 130478


In [ ]:
# sync labels
labels['wall_clock_time'] = pd.to_datetime(labels['wall_clock_time'], utc=True)
labels['accel_elapsed_sec'] = (labels['wall_clock_time'] - accel_start).dt.total_seconds()
print(labels['event_type'].value_counts().to_string())

event_type
NORMAL              248
POTHOLE              56
SPEEDBUMP             3
COLLECTION_START      2
COLLECTION_STOP       2


In [ ]:
# process gpx
points = []
for track in gpx.tracks:
    for segment in track.segments:
        for point in segment.points:
            points.append({'timestamp': point.time, 'lat': point.latitude,
                           'lon': point.longitude, 'elevation': point.elevation})
gps = pd.DataFrame(points)
gps['accel_elapsed_sec'] = (gps['timestamp'] - accel_start).dt.total_seconds()
print(f"gps points: {len(gps)}")



gps points: 1248


In [ ]:
labels.to_csv('synced_up_labels.csv', index=False)
gps.to_csv('synced_up_gps.csv', index=False)
print("created synced_labels.csv and synced_gps.csv")

created synced_labels.csv and synced_gps.csv
